# SpecRoute: Remote Target Model Host for Hybrid Speculative Decoding
### Google Colab & Kaggle Notebook (GPU Accelerated)

This notebook hosts the high-capacity **Target Model** (e.g., `Qwen/Qwen2.5-7B-Instruct` or `meta-llama/Meta-Llama-3.1-8B-Instruct`) on a remote GPU (such as a Google Colab T4/L4 or Kaggle P100/T4).

It exposes:
1. **Single-pass Parallel Verification API** (`POST /v1/speculative/verify`)
2. **Low-latency Persistent WebSocket** (`WS /v1/speculative/ws`)
3. **Tokenizer Parity Synchronization** (`POST /v1/speculative/tokenizer_check`)

The server is tunneled to the public internet via **ngrok**, providing a public URL that your local laptop's `config.yaml` connects to.

## Step 1: Install Dependencies
Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest-asyncio websockets pydantic

## Step 2: Configure Models & Ngrok Credentials

In [ ]:
import os
from getpass import getpass

# Set your target model
TARGET_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # or "meta-llama/Meta-Llama-3.1-8B-Instruct"
LOAD_IN_4BIT = True  # Fits comfortably in <6GB VRAM on free Colab T4 (15GB)

# Enter your free ngrok authentication token (from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "")
if not NGROK_AUTHTOKEN:
    print("Please enter your ngrok authtoken (press Enter to skip if using localtunnel):")
    try:
        NGROK_AUTHTOKEN = getpass()
    except Exception:
        NGROK_AUTHTOKEN = ""

## Step 3: Load Target Model & Shared Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

print(f"Loading tokenizer for {TARGET_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_ID, trust_remote_code=True)

quant_config = None
if LOAD_IN_4BIT and torch.cuda.is_available():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

print(f"Loading weights for target model {TARGET_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    quantization_config=quant_config,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
model.eval()
print("Target model ready for speculative verification!")

## Step 4: Speculative Verification Engine with KV-Cache Truncation

In [ ]:
import time
from typing import Dict, List, Optional

class SessionKVCache:
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.past_key_values = None
        self.current_length = 0
        self.verified_tokens = []

    def crop_to_length(self, target_len: int):
        if self.past_key_values is None:
            self.current_length = target_len
            return
        if hasattr(self.past_key_values, "crop"):
            try:
                self.past_key_values.crop(target_len)
                self.current_length = target_len
                return
            except Exception:
                pass
        try:
            new_pkv = []
            for layer in self.past_key_values:
                k, v = layer[0], layer[1]
                new_pkv.append((k[:, :, :target_len, :], v[:, :, :target_len, :]))
            self.past_key_values = tuple(new_pkv)
            self.current_length = target_len
        except Exception as e:
            print(f"Warning cropping KV cache: {e}")
            self.current_length = target_len

sessions: Dict[str, SessionKVCache] = {}

def verify_draft_sequence(session_id: str, draft_tokens: List[int], temperature: float = 0.0) -> dict:
    t0 = time.perf_counter()
    session = sessions.get(session_id)
    if not session:
        return {"error": f"Session {session_id} not found"}

    k = len(draft_tokens)
    prefix_len = session.current_length
    input_ids = torch.tensor([draft_tokens], device=model.device)

    with torch.no_grad():
        # Single parallel forward pass over all K candidate tokens!
        outputs = model(input_ids=input_ids, past_key_values=session.past_key_values, use_cache=True)
        logits = outputs.logits[0]  # Shape: [K, vocab_size]

    accepted_tokens = []
    rejection_index = None
    corrective_token = None

    for i in range(k):
        pos_logits = logits[i]
        if temperature > 0.0:
            probs = torch.softmax(pos_logits / temperature, dim=-1)
            target_tok = int(torch.multinomial(probs, num_samples=1).item())
        else:
            target_tok = int(torch.argmax(pos_logits, dim=-1).item())

        draft_tok = draft_tokens[i]
        if draft_tok == target_tok:
            accepted_tokens.append(draft_tok)
            if draft_tok == tokenizer.eos_token_id:
                rejection_index = i
                corrective_token = draft_tok
                break
        else:
            rejection_index = i
            corrective_token = target_tok
            break

    if rejection_index is None:
        # All accepted: sample bonus corrective token
        last_logits = logits[k - 1]
        if temperature > 0.0:
            probs = torch.softmax(last_logits / temperature, dim=-1)
            corrective_token = int(torch.multinomial(probs, num_samples=1).item())
        else:
            corrective_token = int(torch.argmax(last_logits, dim=-1).item())

    m = len(accepted_tokens)
    new_kv_len = prefix_len + m
    session.past_key_values = outputs.past_key_values
    session.crop_to_length(new_kv_len)
    session.verified_tokens.extend(accepted_tokens)

    latency_ms = (time.perf_counter() - t0) * 1000.0
    is_eos = (corrective_token == tokenizer.eos_token_id) or (tokenizer.eos_token_id in accepted_tokens)

    return {
        "session_id": session_id,
        "accepted_tokens": accepted_tokens,
        "accepted_count": m,
        "rejection_index": rejection_index,
        "corrective_token": corrective_token,
        "acceptance_rate": (m / k) if k > 0 else 0.0,
        "target_kv_len": new_kv_len,
        "is_eos": is_eos,
        "server_latency_ms": latency_ms,
    }

## Step 5: Build FastAPI Verification Service

In [ ]:
import json
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from pydantic import BaseModel
import nest_asyncio
import uvicorn

nest_asyncio.apply()
app = FastAPI(title="SpecRoute Colab Server")

class PrefillRequest(BaseModel):
    session_id: str
    prompt_token_ids: List[int]
    temperature: float = 0.0

class VerifyReq(BaseModel):
    session_id: str
    draft_tokens: List[int]
    temperature: float = 0.0

@app.get("/health")
def health():
    return {"status": "ok", "device": str(model.device), "sessions": len(sessions)}

@app.post("/v1/speculative/tokenizer_check")
def tokenizer_check(payload: dict):
    client_vocab = payload.get("vocab_size", 0)
    server_vocab = len(tokenizer)
    test_phrase = payload.get("test_phrase", "The quick brown fox jumps over the lazy dog.")
    server_ids = tokenizer.encode(test_phrase, add_special_tokens=False)
    client_ids = payload.get("test_token_ids", [])
    matches = (client_vocab == server_vocab) and (server_ids == client_ids)
    return {
        "status": "ok" if matches else "mismatch",
        "synchronized": matches,
        "server_tokenizer_id": TARGET_MODEL_ID,
        "server_vocab_size": server_vocab,
        "matches_vocab_size": (client_vocab == server_vocab),
        "matches_test_ids": (server_ids == client_ids),
        "message": "Tokenizer verified" if matches else "Tokenizer mismatch detected",
    }

@app.post("/v1/speculative/session/init")
def init_session(req: PrefillRequest):
    sess = SessionKVCache(req.session_id)
    sessions[req.session_id] = sess
    input_ids = torch.tensor([req.prompt_token_ids], device=model.device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, use_cache=True)
        logits = outputs.logits[:, -1, :]
        first_tok = int(torch.argmax(logits, dim=-1).item())
    sess.past_key_values = outputs.past_key_values
    sess.current_length = len(req.prompt_token_ids)
    sess.verified_tokens = list(req.prompt_token_ids) + [first_tok]
    return {
        "session_id": req.session_id,
        "status": "ok",
        "prefill_length": len(req.prompt_token_ids),
        "first_token_id": first_tok,
        "is_eos": (first_tok == tokenizer.eos_token_id),
    }

@app.post("/v1/speculative/verify")
def verify_endpoint(req: VerifyReq):
    return verify_draft_sequence(req.session_id, req.draft_tokens, req.temperature)

@app.websocket("/v1/speculative/ws")
async def ws_endpoint(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            raw = await ws.receive_text()
            msg = json.loads(raw)
            mtype = msg.get("type")
            data = msg.get("data", {})
            if mtype == "init":
                resp = init_session(PrefillRequest(**data))
                await ws.send_text(json.dumps({"type": "init_response", "data": resp}))
            elif mtype == "verify":
                resp = verify_draft_sequence(data["session_id"], data["draft_tokens"], data.get("temperature", 0.0))
                await ws.send_text(json.dumps({"type": "verify_response", "data": resp}))
            elif mtype == "check":
                resp = tokenizer_check(data)
                await ws.send_text(json.dumps({"type": "check_response", "data": resp}))
            elif mtype == "close":
                sessions.pop(data.get("session_id"), None)
                await ws.send_text(json.dumps({"type": "close_response", "data": {"freed": True}}))
    except WebSocketDisconnect:
        pass

## Step 6: Start Ngrok Public Tunnel & Run Server

In [ ]:
from pyngrok import ngrok

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

PORT = 8001
public_url = ngrok.connect(PORT).public_url
print("=" * 70)
print(f"  PUBLIC TUNNEL URL: {public_url}")
print("  Copy the URL above and paste it into your local config.yaml:")
print(f"  remote_server:")
print(f"    base_url: \"{public_url}\"")
print("=" * 70)

# Launch the verification server
uvicorn.run(app, host="0.0.0.0", port=PORT)